In [0]:
env = dbutils.notebook.run("/Workspace/anihan_tech_np/anihan_tech_np/00_Parameters/env_dependencies", 120)
print(env)

In [0]:
spark.sql(f"drop table if exists anihan_tech_{env}.00_master_01_brz.masterdata_company")

In [0]:
path = f"/Volumes/anihan_tech_{env}/00_master_00_lz/lz_vl/company/"


try:
    items = dbutils.fs.ls(path)
    print("Files to be deleted in landing zone volume:")
    display(items)

    if len(items) == 0:
        print(f"Nothing to delete. Directory is already empty: {path}")
    else:
        dbutils.fs.rm(path, True)
        print(f"Successfully deleted: {path}")

except Exception as e:
    print(f"Path does not exist or cannot be accessed: {path}")
    print(f"Error: {e}")

In [0]:
nc_path = path+"new_company"

dbutils.fs.mkdirs(nc_path)
print(f"Folder created: {path}")

In [0]:
#using auto loader
import re
from pyspark.sql.functions import initcap, from_utc_timestamp, current_timestamp, col
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    TimestampType
)

schema = StructType([
    StructField("company_name", StringType(), True),
    StructField("company_id", StringType(), True),
    StructField("join_date", TimestampType(), True),
    StructField("is_active", BooleanType(), True)
])
checkpoint_path = path+"checkpoints/masterdata_company"

df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .schema(schema)
         .load(nc_path)
         .withColumn(
             "company_name",
             initcap("company_name")
         )
)

new_columns = [
    re.sub(r"[ ,;{}()\n\t=]", "_", c)
    for c in df.columns
]

df = df.toDF(*new_columns)
df = (
    df.withColumn("source_file", col("_metadata.file_path"))
      .withColumn("ingested_timestamp_in_ph", from_utc_timestamp(current_timestamp(), "Asia/Manila")))

query = (
    df.writeStream
      .format("delta")
      .outputMode("append")
      .option("checkpointLocation", checkpoint_path)
      .trigger(availableNow=True)
      .toTable(
          f"anihan_tech_{env}.00_master_01_brz.masterdata_company"
      )
)

query.awaitTermination()